# VisA Generalization — PatchCore

**GPU**: ON | **Internet**: ON

Runs all 5 corruption types × 3 severities × 6 rescue methods across 12 VisA categories, 3 seeds. Saves after every phase. Resume-safe — add previous output CSV as input to continue.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
import sys
# ---------------------------------------------------------------------------
# Fix: Disable rich/tqdm progress bars that cause RecursionError on Kaggle.
# ---------------------------------------------------------------------------
os.environ["ANOMALIB_USE_RICH"] = "0"
os.environ["RICH_NO_THEME"] = "1"
sys.setrecursionlimit(5000)
import time
import json
import gc
import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
from torch.utils.data import Dataset


## 0. Global Setup & Timeout Logic

In [ ]:
# ---------------------------------------------------------------------------
# 0. Global Setup & Timeout Logic
# ---------------------------------------------------------------------------
START_TIME = time.time()
CATEGORIES = [
    "candle", "pcb1", "cashew", "pipe_fryum"
]
SEEDS = [42, 123, 456]
TIMEOUT_SECONDS = 11.5 * 3600
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
import shutil
import time
# Target the exact path throwing the error
ORIGINAL_DATA_PATH = "/kaggle/input/datasets/ess1004/visa-anomaly-detection"
# Fallback just in case Kaggle mounted it at the standard root
if not os.path.exists(ORIGINAL_DATA_PATH):
    ORIGINAL_DATA_PATH = "/kaggle/input/visa-anomaly-detection"
# Create a brand new writable path to avoid any cached folder confusion
VISA_ROOT = "/kaggle/working/visa_writable_copy"
# Copy ONLY the target categories to working directory to save massive amounts of Kaggle disk space and time
if not os.path.exists(VISA_ROOT):
    print(f"Copying dataset from {ORIGINAL_DATA_PATH} to {VISA_ROOT}...")
    os.makedirs(VISA_ROOT, exist_ok=True)
    for cat in CATEGORIES:
        src = os.path.join(ORIGINAL_DATA_PATH, cat)
        dst = os.path.join(VISA_ROOT, cat)
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
# ---------------------------------------------------------------------------
# Reorganize VisA to MVTec Layout (inside visa_pytorch/ subfolder)
# ---------------------------------------------------------------------------
def reorganize_visa_to_mvtec(root_dir):
    """Reorganize raw VisA layout into MVTec-style layout inside visa_pytorch/.
    
    The Visa datamodule expects: root/visa_pytorch/{category}/train/good/
    Raw VisA has: root/{category}/Data/Images/Normal/
    
    This function reorganizes the data accordingly. If data is already in
    visa_pytorch/ layout, it's a no-op.
    """
    import os, shutil
    
    split_root = os.path.join(root_dir, "visa_pytorch")
    
    # Skip if already reorganized
    if os.path.exists(split_root):
        print(f"visa_pytorch/ already exists at {split_root}, skipping reorganization.")
        return
    
    os.makedirs(split_root, exist_ok=True)
    
    for cat in sorted(os.listdir(root_dir)):
        cat_dir = os.path.join(root_dir, cat)
        if not os.path.isdir(cat_dir): continue
        if cat == "visa_pytorch": continue
        
        data_dir = os.path.join(cat_dir, "Data")
        if not os.path.exists(data_dir):
            # Already in MVTec layout? Just move into visa_pytorch/
            if os.path.exists(os.path.join(cat_dir, "train")):
                shutil.move(cat_dir, os.path.join(split_root, cat))
                print(f"Moved pre-organized {cat} into visa_pytorch/")
            continue
        
        print(f"Reorganizing {cat} to MVTec layout...")
        dst_cat = os.path.join(split_root, cat)
        
        # 1. Normal images -> train/good
        normal_dir = os.path.join(data_dir, "Images", "Normal")
        train_good = os.path.join(dst_cat, "train", "good")
        if os.path.exists(normal_dir):
            os.makedirs(os.path.dirname(train_good), exist_ok=True)
            shutil.move(normal_dir, train_good)
            
        # 2. Anomaly images -> test/bad
        anomaly_dir = os.path.join(data_dir, "Images", "Anomaly")
        test_bad = os.path.join(dst_cat, "test", "bad")
        if os.path.exists(anomaly_dir):
            os.makedirs(test_bad, exist_ok=True)
            for item in os.listdir(anomaly_dir):
                shutil.move(os.path.join(anomaly_dir, item), os.path.join(test_bad, item))
                
        # 3. Masks -> ground_truth/bad
        mask_dir = os.path.join(data_dir, "Masks", "Anomaly")
        gt_bad = os.path.join(dst_cat, "ground_truth", "bad")
        if os.path.exists(mask_dir):
            os.makedirs(gt_bad, exist_ok=True)
            for item in os.listdir(mask_dir):
                shutil.move(os.path.join(mask_dir, item), os.path.join(gt_bad, item))
                
        # 4. Create empty test/good folder
        os.makedirs(os.path.join(dst_cat, "test", "good"), exist_ok=True)
        
        # 5. Cleanup original category folder
        shutil.rmtree(cat_dir)
        
    print(f"Reorganization complete. Categories in: {split_root}")
reorganize_visa_to_mvtec(VISA_ROOT)
print("Dataset successfully copied to writable directory!")
CONFIG_PATH = "/kaggle/input/notebooks/hasanmahmudabdullah/03-severity-calibration/experiment_config.json"
OUTPUT_FILE = "/kaggle/working/results/visa_patchcore_augmented.csv"
PARTIAL_FILE = "/kaggle/working/results/visa_patchcore_augmented_partial.csv"
print(f"Script started at {time.ctime(START_TIME)}")
print(f"Graceful timeout set to {TIMEOUT_SECONDS / 3600:.1f} hours.")
print(f"Using Writable Dataset Root: {VISA_ROOT}")
def save_results():
    if 'all_results' in globals() and all_results:
        os.makedirs("/kaggle/working/results", exist_ok=True)
        df = pd.DataFrame(all_results)
        df.to_csv(OUTPUT_FILE, index=False)
        df.to_csv(PARTIAL_FILE, index=False)
    
    # Check timeout after every save to prevent Kaggle 12-hour hard-kill
    elapsed = time.time() - START_TIME
    if elapsed > TIMEOUT_SECONDS:
        print(f"\n{'!'*60}\nTIMEOUT REACHED ({elapsed/3600:.1f}h). Exiting gracefully.\n{'!'*60}")
        import sys
        sys.exit(0)
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 1. Dependency Check — same pattern as run_patchcore.py

In [ ]:
# ---------------------------------------------------------------------------
# 1. Dependency Check
# ---------------------------------------------------------------------------
def ensure_dependencies():
    import subprocess
    import sys
    packages = ["anomalib", "lightning", "albumentationsx", "scikit-image", "opencv-python-headless"]
    for package in packages:
        try:
            check_name = "cv2" if package == "opencv-python-headless" else (
                package.replace("-", "_") if package != "albumentationsx" else "albumentations")
            __import__(check_name)
        except ImportError:
            print(f"Installing missing dependency: {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
ensure_dependencies()
# Now safe to import
import lightning as L
from anomalib.engine import Engine
from anomalib.models import Patchcore, Padim
from anomalib.data import Visa as VisaDataModule


## 2. Corruption & Rescue Functions — identical to run_patchcore.py

In [ ]:
# ---------------------------------------------------------------------------
# 2. Corruption & Rescue Functions — identical to run_patchcore.py
# ---------------------------------------------------------------------------
def apply_low_light(image, gamma, seed=42):
    rng = np.random.default_rng(seed)
    img_float = image.astype(np.float64) / 255.0
    img_dark = np.power(img_float, 1.0 / gamma)
    img_noisy = rng.poisson(np.clip(img_dark * 50.0, 0, None)) / 50.0
    return np.clip(img_noisy * 255, 0, 255).astype(np.uint8)
def apply_gaussian_blur(image, sigma, kernel_size):
    return cv2.GaussianBlur(image, (kernel_size, kernel_size), sigmaX=sigma)
def apply_motion_blur(image, kernel_size):
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size - 1) / 2), :] = np.ones(kernel_size)
    kernel /= kernel_size
    return cv2.filter2D(image, -1, kernel)
def apply_sensor_noise(image, gauss_var, seed=42):
    """Apply Gaussian noise + 5% salt-and-pepper impulse noise.
    
    Note: The salt-and-pepper component (sp_ratio=0.05) adds impulse noise
    on top of the Gaussian noise. This means NLM denoising is suboptimal
    for this corruption type; a median filter would be more appropriate
    for the impulse component.
    """
    rng = np.random.default_rng(seed)
    img_float = image.astype(np.float64) / 255.0
    noise = rng.normal(0, np.sqrt(gauss_var), img_float.shape)
    img_noisy = img_float + noise
    sp_ratio = 0.05
    salt = rng.random(img_float.shape[:2]) < (sp_ratio / 2)
    pepper = rng.random(img_float.shape[:2]) < (sp_ratio / 2)
    img_noisy[salt] = 1.0
    img_noisy[pepper] = 0.0
    return np.clip(img_noisy * 255, 0, 255).astype(np.uint8)
def apply_fog(image, fog_coef_lower, fog_coef_upper, alpha_coef=0.1, seed=42):
    """Apply synthetic fog/haze using Albumentations RandomFog with deterministic seed."""
    import random as _random
    _random.seed(seed)
    np.random.seed(seed % (2**31))
    t = A.RandomFog(fog_coef_range=(fog_coef_lower, fog_coef_upper),
                    alpha_coef=alpha_coef, p=1.0)
    return t(image=image)["image"]
def apply_corruption(image, ctype, severity, config, seed=42):
    params = config["corruptions"][ctype][severity]
    if ctype == "low_light":     return apply_low_light(image, params["gamma"], seed)
    elif ctype == "gaussian_blur": return apply_gaussian_blur(image, params["sigma"], params["kernel_size"])
    elif ctype == "motion_blur":   return apply_motion_blur(image, params["kernel_size"])
    elif ctype == "sensor_noise":  return apply_sensor_noise(image, params["gauss_var"], seed)
    elif ctype == "fog_haze":      return apply_fog(image, params["fog_coef_lower"], params["fog_coef_upper"], params.get("alpha_coef", 0.1))
    else: raise ValueError(f"Unknown corruption type: {ctype}")
def apply_clahe(image, clip_limit=3.0, tile_grid_size=(8, 8)):
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size).apply(l)
    return cv2.cvtColor(cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
def apply_wiener_deconv(image, sigma, kernel_size, balance=0.1):
    from skimage.restoration import wiener
    from skimage import img_as_float, img_as_ubyte
    img_float = img_as_float(image)
    ax = np.arange(-kernel_size // 2 + 1, kernel_size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    psf = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    psf /= psf.sum()
    result = np.zeros_like(img_float)
    for c in range(3): result[:, :, c] = wiener(img_float[:, :, c], psf, balance)
    return img_as_ubyte(np.clip(result, 0, 1))
def apply_motion_wiener_deconv(image, kernel_size, balance=0.1):
    from skimage.restoration import wiener
    from skimage import img_as_float, img_as_ubyte
    img_float = img_as_float(image)
    psf = np.zeros((kernel_size, kernel_size))
    psf[kernel_size // 2, :] = 1.0 / kernel_size
    result = np.zeros_like(img_float)
    for c in range(3): result[:, :, c] = wiener(img_float[:, :, c], psf, balance)
    return img_as_ubyte(np.clip(result, 0, 1))
def apply_nlm_denoise(image, patch_size=7, patch_distance=11):
    from skimage.restoration import denoise_nl_means, estimate_sigma
    img_float = image.astype(np.float64) / 255.0
    sigma_est = np.mean(estimate_sigma(img_float, channel_axis=-1))
    denoised = denoise_nl_means(img_float, h=1.15 * sigma_est, patch_size=patch_size,
                                patch_distance=patch_distance, fast_mode=True, channel_axis=-1)
    return np.clip(denoised * 255, 0, 255).astype(np.uint8)
def apply_retinex(image):
    img_float = image.astype(np.float64) + 1.0
    blurred = cv2.GaussianBlur(img_float, (0, 0), 30)
    log_retinex = np.log10(img_float) - np.log10(blurred + 1.0)
    for i in range(3):
        log_retinex[:, :, i] = (log_retinex[:, :, i] - np.min(log_retinex[:, :, i])) / \
                                 (np.max(log_retinex[:, :, i]) - np.min(log_retinex[:, :, i])) * 255
    return log_retinex.astype(np.uint8)
def apply_dark_channel_prior_dehaze(image, omega=0.95, patch_size=15):
    from scipy.ndimage import minimum_filter
    img = image.astype(np.float64) / 255.0
    dark = minimum_filter(np.min(img, axis=2), size=patch_size)
    flat = dark.ravel()
    top = np.argsort(flat)[-max(1, int(0.001 * len(flat))):]
    A = np.max(img.reshape(-1, 3)[top], axis=0)
    normed_dark = minimum_filter(np.min(img / (A + 1e-6), axis=2), size=patch_size)
    t = np.clip(1 - omega * normed_dark, 0.1, 1.0)
    result = (img - A) / t[:, :, np.newaxis] + A
    return np.clip(result * 255, 0, 255).astype(np.uint8)
def get_rescue_map(severity, config):
    """Return rescue methods with PSF parameters matched to the current corruption severity.
    
    This ensures Wiener deconvolution uses the exact PSF that generated the blur,
    rather than a hardcoded severe-tier kernel.
    """
    gauss_p = config["corruptions"]["gaussian_blur"][severity]
    motion_p = config["corruptions"]["motion_blur"][severity]
    return {
        "low_light":     [("CLAHE", apply_clahe), ("Retinex", apply_retinex)],
        "gaussian_blur": [("Wiener", lambda img, s=gauss_p["sigma"], k=gauss_p["kernel_size"]:
                           apply_wiener_deconv(img, sigma=s, kernel_size=k))],
        "motion_blur":   [("Wiener (Motion PSF)", lambda img, k=motion_p["kernel_size"]:
                         apply_motion_wiener_deconv(img, kernel_size=k))],
        "sensor_noise":  [("NLM Denoise", apply_nlm_denoise)],
        "fog_haze":      [("Dehaze (Dark Channel)", apply_dark_channel_prior_dehaze)],
    }


## 3. Load Configuration

In [ ]:
# ---------------------------------------------------------------------------
# 3. Load Configuration
# ---------------------------------------------------------------------------
for cfg_path in [CONFIG_PATH, "experiment_config.json"]:
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            config = json.load(f)
        print(f"Config loaded from: {cfg_path}")
        break
else:
    raise FileNotFoundError(f"experiment_config.json not found. Add severity calibration output as Kaggle input.")


## 4. Corrupted Dataset Wrapper — identical to run_patchcore.py

In [ ]:
# ---------------------------------------------------------------------------
# 4. Corrupted Dataset Wrapper — identical to run_patchcore.py
# ---------------------------------------------------------------------------
class CorruptedDatasetWrapper(Dataset):
    def __init__(self, base_dataset, ctype, severity, config, rescue_func=None):
        self.base_dataset = base_dataset
        self.ctype = ctype
        self.severity = severity
        self.config = config
        self.rescue_func = rescue_func
    def __len__(self): return len(self.base_dataset)
    def __getattr__(self, name):
        return getattr(self.base_dataset, name)
    def __getitem__(self, idx):
        import dataclasses
        item = self.base_dataset[idx]
        if dataclasses.is_dataclass(item): image = item.image
        else: image = item["image"]
        if isinstance(image, torch.Tensor):
            img_np = image.permute(1, 2, 0).cpu().numpy()
            if img_np.max() <= 1.0: img_np = (img_np * 255).astype(np.uint8)
            else: img_np = img_np.astype(np.uint8)
        else:
            img_np = np.array(image).astype(np.uint8)
            
        corrupted = apply_corruption(img_np, self.ctype, self.severity, self.config, seed=42 + idx)
        final_img = self.rescue_func(corrupted) if self.rescue_func else corrupted
        final_tensor = torch.from_numpy(final_img).permute(2, 0, 1).float() / 255.0
        if dataclasses.is_dataclass(item): return dataclasses.replace(item, image=final_tensor)
        else: item["image"] = final_tensor; return item


## 5. Engine & Resume Logic — identical to run_patchcore.py

In [ ]:
class DisableCheckpointing(L.Callback):
    def setup(self, trainer, pl_module, stage):
        from lightning.pytorch.callbacks import ModelCheckpoint
        trainer.callbacks = [cb for cb in trainer.callbacks if not isinstance(cb, ModelCheckpoint)]

def make_engine():
    return Engine(max_epochs=1, accelerator="auto", devices=1,
                  default_root_dir="/tmp/anomalib", enable_progress_bar=False,
                  callbacks=[DisableCheckpointing()])

def safe_auroc(result_dict):
    for key in ["image_AUROC", "image_auroc", "auroc", "AUROC", "test_image_AUROC"]:
        if key in result_dict: return result_dict[key]
    print(f"  ⚠️  AUROC key not found. Keys: {list(result_dict.keys())}")
    return None

completed_seeds = set()
completed_rows = set()
all_results = []
os.makedirs("results", exist_ok=True)

import glob
import shutil

input_partials = glob.glob("/kaggle/input/**/*partial*.csv", recursive=True)
if input_partials:
    latest_partial = max(input_partials, key=os.path.getmtime)
    os.makedirs(os.path.dirname(PARTIAL_FILE), exist_ok=True)
    shutil.copy(latest_partial, PARTIAL_FILE)
    print(f"Auto-restored partial progress from: {latest_partial}")

for p in [OUTPUT_FILE, PARTIAL_FILE]:
    if os.path.exists(p):
        try:
            old_df = pd.read_csv(p)
            counts = old_df.groupby(['category', 'seed']).size()
            
            for (cat, seed), count in counts.items():
                if count >= 34: 
                    completed_seeds.add((cat, int(seed)))
            
            all_results = old_df.to_dict("records")
            for r in all_results:
                completed_rows.add((r['category'], int(r['seed']), r['phase'], r['ctype'], r['severity'], r['rescue']))
            
            print(f"Resumed {len(completed_seeds)} fully completed seeds and {len(completed_rows)} individual rows from {p}.")
            break
        except Exception as e:
            print(f"Could not load existing results: {e}")



## 6. Main Loop

In [ ]:
import random
import cv2
import shutil
import sys
_AUG_CTYPES = ["low_light", "gaussian_blur", "motion_blur", "sensor_noise", "fog_haze"]
_AUG_SEVS = ["mild", "moderate", "severe"]
AUG_VISA_ROOT = "/kaggle/working/visa_augmented"
def prepare_augmented_train_data(category, aug_prob=0.5, rng_seed=0):
    random.seed(rng_seed)
    print(f"Pre-generating augmented training data (aug_prob={aug_prob})...")
    
    cat_src = os.path.join(VISA_ROOT, "visa_pytorch", category)
    cat_dst = os.path.join(AUG_VISA_ROOT, "visa_pytorch", category)
    
    if os.path.exists(cat_dst):
        shutil.rmtree(cat_dst)
        
    print(f"Copying exact dataset structure from {cat_src} to {cat_dst}...")
    shutil.copytree(cat_src, cat_dst)
    
    augmented_count = 0
    total_count = 0
    for dirpath, dirnames, filenames in os.walk(cat_dst):
        folder_name = os.path.basename(dirpath)
        if folder_name not in ("Normal", "good"):
            continue
        for fname in sorted(filenames):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")): continue
            total_count += 1
            if random.random() < aug_prob:
                img_path = os.path.join(dirpath, fname)
                img_bgr = cv2.imread(img_path)
                if img_bgr is None: continue
                img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                ctype = random.choice(_AUG_CTYPES)
                sev   = random.choice(_AUG_SEVS)
                img = apply_corruption(img, ctype, sev, config, seed=rng_seed)
                cv2.imwrite(img_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
                augmented_count += 1
                
    print(f"Augmented {augmented_count}/{total_count} Normal/good images successfully.\n")
print("=" * 60)
print("VALIDATION: Checking dataset directories...")
if not os.path.isdir(VISA_ROOT):
    raise FileNotFoundError(f"FATAL: VISA_ROOT does not exist: {VISA_ROOT}")
# Quick sanity check
print("=" * 60)


In [ ]:
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VisA root: {VISA_ROOT}")
print(f"Categories: {CATEGORIES}")
for category in CATEGORIES:
    for seed in SEEDS:
        
        if (category, int(seed)) in completed_seeds:
            print(f"⏩ Skipping fully completed seed: {category} (Seed {seed})")
            continue
            
        print(f"\n{'='*60}\nCATEGORY: {category.upper()} | SEED: {seed}\n{'='*60}")
        L.seed_everything(seed)
        prepare_augmented_train_data(category, aug_prob=0.5, rng_seed=seed)
        
        engine = make_engine()
        model = Patchcore(backbone="wide_resnet50_2", num_neighbors=9)
        datamodule = VisaDataModule(root=AUG_VISA_ROOT, category=category, train_batch_size=32, eval_batch_size=32)
        
        try:
            # We MUST fit the model to build the coreset for testing. Takes ~5-10 mins.
            print(f"[PHASE 1] Training Baseline...")
            engine.fit(model=model, datamodule=datamodule)
            
            if (category, int(seed), "baseline", "clean", "none", "none") not in completed_rows:
                res = engine.test(model=model, datamodule=datamodule)
                clean_auroc = safe_auroc(res[0])
                all_results.append({"model": "PatchCore", "dataset": "VisA", "category": category,
                                     "seed": seed, "phase": "baseline", "ctype": "clean",
                                     "severity": "none", "rescue": "none", "image_AUROC": clean_auroc})
                save_results()
                completed_rows.add((category, int(seed), "baseline", "clean", "none", "none"))
            else:
                print(f"⏩ Skipping Baseline Test (Already in partial)")
            dm_test = VisaDataModule(root=VISA_ROOT, category=category, train_batch_size=32, eval_batch_size=32)
            dm_test.setup(stage="test")
            from torch.utils.data import DataLoader
            for ctype, severities in config["corruptions"].items():
                for sev in severities.keys():
                    if (category, int(seed), "degradation", ctype, sev, "none") not in completed_rows:
                        print(f"[PHASE 2] Degradation: {ctype} ({sev})")
                        deg_loader = DataLoader(
                            CorruptedDatasetWrapper(dm_test.test_data, ctype, sev, config),
                            batch_size=32, num_workers=4, collate_fn=dm_test.test_data.collate_fn)
                        res = engine.test(model=model, dataloaders=deg_loader)
                        all_results.append({"model": "PatchCore", "dataset": "VisA", "category": category,
                                            "seed": seed, "phase": "degradation", "ctype": ctype,
                                            "severity": sev, "rescue": "none", "image_AUROC": safe_auroc(res[0])})
                        save_results()
                        completed_rows.add((category, int(seed), "degradation", ctype, sev, "none"))
                    else:
                        print(f"⏩ Skipping Degradation: {ctype} ({sev})")
                    if ctype in get_rescue_map(sev, config):
                        for r_name, r_func in get_rescue_map(sev, config)[ctype]:
                            if (category, int(seed), "rescue", ctype, sev, r_name) not in completed_rows:
                                print(f"[PHASE 3] Rescue: {ctype} ({sev}) + {r_name}")
                                res_loader = DataLoader(
                                    CorruptedDatasetWrapper(dm_test.test_data, ctype, sev, config, rescue_func=r_func),
                                    batch_size=32, num_workers=4, collate_fn=dm_test.test_data.collate_fn)
                                res = engine.test(model=model, dataloaders=res_loader)
                                all_results.append({"model": "PatchCore", "dataset": "VisA", "category": category,
                                                    "seed": seed, "phase": "rescue", "ctype": ctype,
                                                    "severity": sev, "rescue": r_name,
                                                    "image_AUROC": safe_auroc(res[0])})
                                save_results()
                                completed_rows.add((category, int(seed), "rescue", ctype, sev, r_name))
                            else:
                                print(f"⏩ Skipping Rescue: {ctype} ({sev}) + {r_name}")
        except Exception as e:
            print(f"❌ FATAL Error in {category}/{seed}: {e}")
            save_results()
            raise
        del model
        del engine
        cleanup_memory()
save_results()
print(f"\n{'='*60}\nVISA COMPLETE — {len(all_results)} rows saved.\n{'='*60}")
